# MITONet - Red River

In [ ]:
import os
import sys
import tensorflow as tf
tf.keras.backend.set_floatx('float64') 


import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.ticker import LinearLocator, ScalarFormatter, FormatStrFormatter
from matplotlib import ticker as mtick
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import cmocean

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error as mae
import joblib

from IPython.display import display
import matplotlib.ticker as ticker
from matplotlib import rcParams
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import netCDF4 as nc
from tqdm import tqdm

import pyproj
from pyproj import Transformer
from functools import partial

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.patches as mpatches

plt.rc('font', family='serif')
plt.rcParams.update({'font.size': 20,
                     'lines.linewidth': 2,
                     'lines.markersize':10,
                     'axes.labelsize': 16, # fontsize for x and y labels (was 10)
                     'axes.titlesize': 20,
                     'xtick.labelsize': 16,
                     'ytick.labelsize': 16,
                     'legend.fontsize': 16,
                     'axes.linewidth': 2})

import itertools
colors = itertools.cycle(['r','g','b','m','y','c'])
markers = itertools.cycle(['p','d','o','^','s','x',]) #'D','H','v','*'])

from pathlib import Path
try:
    work_dir.exists()
except NameError:
    curr_dir = Path().resolve()
    work_dir = curr_dir.parent  

scripts_dir = work_dir / "src"
settings_dir = work_dir / "settings"
sys.path.append(str(scripts_dir.absolute()))
sys.path.append(str(settings_dir.absolute()))

ae_model_dir = work_dir / "saved_models"
mito_model_dir = work_dir / "saved_models"
scalers_dir = work_dir / "scalers"

import mitonet as mito
import autoencoder as ae
import data_loader as dl 
import processing_utils as pu
import settings_optuna as sett

#Download and copy your data to this folder
data_dir = Path(sett.data_dir)

from importlib import reload as reload

%matplotlib inline

## Specify training configuraiton for scaling purposes and testing cases for visualization

In [ ]:
# Training data for scaling
train_days0 = [15,30,45] 
train_days1 = [20,35,50] 
param_train = [0.024, 0.024125, 0.02425,  0.0245, 0.024625, 0.02475, 0.025,  0.025125, 0.025375, 0.0255,  0.02575, 0.025875, 0.026]
scaling = True
t_skip = sett.t_skip

# Test data
test_days = [5., 60.]
# param_list = [0.02375, 0.023875, 0.024, 0.024125, 0.02425, 0.024375, 0.0245, 0.024625, 0.02475, 0.024875, 0.025, 0.025125, 0.02525, 
#                    0.025375, 0.0255, 0.025625, 0.02575, 0.025875, 0.026, 0.026125, 0.02625]
param_list_test = [0.02375, 0.024875, 0.02625]
unseen_param_loc = [0,9,-1]

## Load Models

In [ ]:
# Load H Models
ae_model_path_dep = str(ae_model_dir) + '/AE_S_dep'
ae_model_dep,_ = ae.load_model(ae_model_path_dep, comp = False)
mito_model_path_dep = str(mito_model_dir) + '/MITO_S_dep'
mito_model_dep = tf.keras.models.load_model(mito_model_path_dep)

# Load U Models
ae_model_path_vx = str(ae_model_dir) + '/AE_S_vx'
ae_model_vx,_ = ae.load_model(ae_model_path_vx, comp = False)
mito_model_path_vx = str(mito_model_dir) + '/MITO_S_vx'
mito_model_vx = tf.keras.models.load_model(mito_model_path_vx)

# Load V Models
ae_model_path_vy = str(ae_model_dir) + '/AE_S_vy'
ae_model_vy,_ = ae.load_model(ae_model_path_vy, comp = False)
mito_model_path_vy = str(mito_model_dir) + '/MITO_S_vy'
mito_model_vy = tf.keras.models.load_model(mito_model_path_vy)

### Print Model Summary

In [ ]:
# Print AE model summary
ae_model_dep.summary()
ae_model_vx.summary()
ae_model_vy.summary()

In [ ]:
# Print MITONet model summary
mito_model_dep.summary()
mito_model_vx.summary()
mito_model_vy.summary()

## Load and visualize mesh, bathymetry and sensor locations

In [ ]:
# Load mesh
mesh_fname = "red_river_mesh.npz"
mesh = dl.load_mesh_adh(data_dir/mesh_fname)

# Extract fields
triangles = mesh['triangles']  # if your indexing in 'triangles' starts at 1, convert to 0-based
lon = mesh['nodes'][:, 0]
lat = mesh['nodes'][:, 1]
elevation = mesh['nodes'][:, 2]

In [ ]:
def UTM_to_LatLon(easting, northing, zone_number, zone_letter):
    """
    Convert UTM coordinates to latitude and longitude.

    Parameters:
    easting (float): The easting (x) coordinate in meters.
    northing (float): The northing (y) coordinate in meters.
    zone_number (int): The UTM zone number.
    zone_letter (str): The UTM zone letter (N for Northern Hemisphere, S for Southern Hemisphere).

    Returns:
    tuple: A tuple containing the latitude and longitude in degrees.
    """
    # Create a Proj object for the UTM coordinate system
    # The '+zone=' argument specifies the UTM zone number.
    # The '+south' argument is needed if the zone is in the Southern Hemisphere.
    # The '+ellps=WGS84 +datum=WGS84' specifies the WGS84 ellipsoid and datum.
    # The '+units=m +no_defs' specifies meters as units and no default parameters.
    utm_proj = pyproj.CRS(f"+proj=utm +zone={zone_number}{' +south' if zone_letter.upper() in ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K', 'L', 'M'] else ''} +ellps=WGS84 +datum=WGS84 +units=m +no_defs")

    # Create a Proj object for the WGS84 geographic coordinate system (latitude/longitude)
    wgs84_proj = pyproj.CRS("WGS84")  # EPSG:4326 is the code for WGS84 lat/lon

    # Perform the transformation
    # transformer = partial(pyproj.transform, utm_proj, wgs84_proj)
    transformer = Transformer.from_crs(utm_proj, wgs84_proj, always_xy=True)
    lon, lat = transformer.transform(easting, northing)

    # print(f"UTM coordinates: Easting={easting}, Northing={northing}, Zone={zone_number}{zone_letter}")
    # print(f"Latitude: {lat}, Longitude: {lon}")

    return lat, lon

# Define the UTM coordinates
easting = mesh['nodes'][:,0].copy()  # Example easting value in meters
northing = mesh['nodes'][:,1].copy()  # Example northing value in meters
zone_number = 15
zone_letter = 'N'  # Or 'N' for Northern Hemisphere, 'S' for Southern Hemisphere

lat, lon = UTM_to_LatLon(easting, northing, zone_number, zone_letter)
print(f"Latitude: {lat}, Longitude: {lon}")

In [ ]:
sensors = [2420,5000,11450]
s1_color = cmocean.cm.phase(0.6)
s2_color = cmocean.cm.phase(1)
s3_color = cmocean.cm.phase(0.8)

valmax = np.nanmax(elevation)
valmin = np.nanmin(elevation)
# disp_min, disp_max = np.nanpercentile(elevation, [10, 90])

extent = [-91.745, -91.665, 31.165, 31.215]

display_crs = ccrs.PlateCarree()      # fallback

fig = plt.figure(figsize=(15, 10))
ax = plt.axes(projection=display_crs)

wmts_vxrl = "https://services.arcgisonline.com/arcgis/rest/services/World_Imagery/MapServer/WMTS/1.0.0/WMTSCapabilities.xml"
ax.add_wmts(wmts_vxrl, layer_name="World_Imagery", zorder=0)

cmap = cmocean.cm.balance

mesh_plot = ax.tripcolor(
    lon, lat, 
    triangles, 
    elevation,
    cmap=cmap,
    edgecolor='k',
    linewidth=0.1,
    vmin=valmin, 
    vmax=valmax,
    transform=ccrs.PlateCarree(),  # your data are lon/lat
)

ax.set_extent(extent, crs=ccrs.PlateCarree())

# mesh_plot.set_rasterized(True)

ax.set_xticks([-91.73, -91.705, -91.68], crs=ccrs.PlateCarree()) 
ax.set_yticks([31.17, 31.19, 31.21], crs=ccrs.PlateCarree())
ax.xaxis.set_major_formatter(LongitudeFormatter(degree_symbol='°'))
ax.yaxis.set_major_formatter(LatitudeFormatter(degree_symbol='°'))

ax.scatter(lon[sensors[0]], lat[sensors[0]], s=150, label='Sensor 1', marker='o',color=s1_color)
ax.scatter(lon[sensors[1]], lat[sensors[1]], s=150, label='Sensor 2', marker='o',color=s2_color)
ax.scatter(lon[sensors[2]], lat[sensors[2]], s=150, label='Sensor 3', marker='o',color=s3_color)

cbar = plt.colorbar(mesh_plot, ax=ax, shrink=0.75)
cbar.set_label('Elevation  (m)')

plt.legend(loc='upper right')
plt.show()

plt.tight_layout()
plt.show()

In [ ]:
d_nodes = sett.discharge_nodes 
t_nodes = sett.tailwater_nodes 
x_series_temp = dl.load_variables_adh(data_dir/'Inset_T5.198e+06_mn0.026250.npz', comp_names=['S_dep'], t_start = 480)['time'][::t_skip].copy() ## Ignoring the first five days of simulation data 
dis_bc = np.load(data_dir / 'red_river_inflow_discharge_T60d.npz')['bc'][:,480:] 
data_dep = dl.load_variables_adh(data_dir/'Inset_T5.198e+06_mn0.026250.npz', comp_names=['S_dep'], t_start = 480)['S_dep'] ## Ignoring the first five days of simulation data 
t_start = np.where(x_series_temp/60/60/24==test_days[0])[0][0] 
x_series = x_series_temp[t_start:]/60/60/24

train_snaps1 = [np.where(x_series_temp/60/60/24==train_days0[0])[0][0], np.where(x_series_temp/60/60/24==train_days1[0])[0][0]] 
train_snaps2 = [np.where(x_series_temp/60/60/24==train_days0[1])[0][0], np.where(x_series_temp/60/60/24==train_days1[1])[0][0]] 
train_snaps3 = [np.where(x_series_temp/60/60/24==train_days0[2])[0][0], np.where(x_series_temp/60/60/24==train_days1[2])[0][0]] 
x_train_lower1 = x_series_temp[train_snaps1[0]]/3600/24 
x_train_lower2 = x_series_temp[train_snaps2[0]]/3600/24 
x_train_lower3 = x_series_temp[train_snaps3[0]]/3600/24 
x_train_upper1 = x_series_temp[train_snaps1[1]]/3600/24 
x_train_upper2 = x_series_temp[train_snaps2[1]]/3600/24 
x_train_upper3 = x_series_temp[train_snaps3[1]]/3600/24

fig = plt.figure(figsize=(15, 6))
gs = gridspec.GridSpec(
    nrows=1,
    ncols=2,
    width_ratios=[1.3, 1],
    wspace=0.25
)

######################
# (a) Bathymetry/map panel
######################
ax_map = fig.add_subplot(gs[0, 0], projection=display_crs)

wmts_vxrl = (
    "https://services.arcgisonline.com/arcgis/rest/services/"
    "World_Imagery/MapServer/WMTS/1.0.0/WMTSCapabilities.xml"
)
ax_map.add_wmts(wmts_vxrl, layer_name="World_Imagery", zorder=0)

cmap = cmocean.cm.balance

mesh_plot = ax_map.tripcolor(
    lon,
    lat,
    triangles,
    elevation,
    cmap=cmap,
    edgecolor='k',
    linewidth=0.01,
    vmin=valmin,
    vmax=valmax,
    transform=ccrs.PlateCarree(),
)

ax_map.set_extent(extent, crs=ccrs.PlateCarree())

ax_map.set_xticks([-91.73, -91.705, -91.68], crs=ccrs.PlateCarree())
ax_map.set_yticks([31.17, 31.19, 31.21], crs=ccrs.PlateCarree())
ax_map.xaxis.set_major_formatter(LongitudeFormatter(degree_symbol='°'))
ax_map.yaxis.set_major_formatter(LatitudeFormatter(degree_symbol='°'))

ax_map.scatter(
    lon[sensors[0]], lat[sensors[0]],
    s=50, label='Sensor 1',
    marker='o', color=s1_color,
    transform=ccrs.PlateCarree()
)
ax_map.scatter(
    lon[sensors[1]], lat[sensors[1]],
    s=50, label='Sensor 2',
    marker='o', color=s2_color,
    transform=ccrs.PlateCarree()
)
ax_map.scatter(
    lon[sensors[2]], lat[sensors[2]],
    s=50, label='Sensor 3',
    marker='o', color=s3_color,
    transform=ccrs.PlateCarree()
)

cbar = fig.colorbar(
    mesh_plot,
    ax=ax_map,
    shrink=0.6,
    orientation='vertical',
)
cbar.set_label('Elevation (m)', fontsize=16)
cbar.ax.tick_params(labelsize=12)

# ax_map.legend(loc='lower left', fontsize=16)

ax_map.text(
    lon[sensors[0]], lat[sensors[0]]-0.001, 'Sensor 1',
    ha='center',va='top',
    fontsize=14, fontweight='bold',
    color='white',
)

ax_map.text(
    lon[sensors[1]], lat[sensors[1]]+0.001, 'Sensor 2',
    ha='center',va='bottom',
    fontsize=14, fontweight='bold',
    color='white',
)

ax_map.text(
    lon[sensors[2]]-0.002, lat[sensors[2]], 'Sensor 3',
    ha='right',va='center',
    fontsize=14, fontweight='bold',
    color='white',
)

ax_map.text(
    0.02, 0.98, '(a)',
    transform=ax_map.transAxes,
    ha='left', va='top',
    fontsize=16, fontweight='bold',
    color='white',
)

ax_map.tick_params(axis='both', which='major', labelsize=16)

#########################
# (b) Inflow / Tailwater panel
#########################
ax1 = fig.add_subplot(gs[0, 1])

ax1.plot(
    x_series,
    dis_bc[0, t_start:-1][::t_skip],
    label='Inflow'
)
ax1.set_ylabel('Inflow Discharge ($m^2/s$)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()
ax2.plot(
    x_series,
    data_dep[t_nodes[1], :-1][::t_skip],
    color='red',
    label='Tailwater'
)
ax2.set_ylabel('Tailwater Elevation ($m$)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

ax1.set_xlabel('Time (days)')

# training windows
ax2.axvline(x_train_lower1, color='black', linestyle='--')
ax2.axvline(x_train_upper1, color='black', linestyle='--')
ax2.axvspan(x_train_lower1, x_train_upper1, color='grey', alpha=0.10, zorder=-1)

ax2.axvline(x_train_lower2, color='black', linestyle='--')
ax2.axvline(x_train_upper2, color='black', linestyle='--')
ax2.axvspan(x_train_lower2, x_train_upper2, color='grey', alpha=0.10, zorder=-1)

ax2.axvline(x_train_lower3, color='black', linestyle='--')
ax2.axvline(x_train_upper3, color='black', linestyle='--')
ax2.axvspan(x_train_lower3, x_train_upper3, color='grey', alpha=0.10, zorder=-1)

ax1.text(
    0.02, 0.98, '(b)',
    transform=ax1.transAxes,
    ha='left', va='top',
    fontsize=16, fontweight='bold'
)

# left axis (discharge)
ax1.tick_params(axis='both', which='major', labelsize=16)
ax1.set_ylabel('Inflow Discharge ($m^2/s$)', color='blue', fontsize=16)
ax1.set_xlabel('Time (days)', fontsize=16)

# right axis (tailwater)
ax2.tick_params(axis='y', which='major', labelsize=16, labelcolor='red')
ax2.set_ylabel('Tailwater Elevation ($m$)', color='red', fontsize=16)

######################################
# --- match heights of the two panels
######################################

# draw once so positions are finalized
fig.canvas.draw()

# get the position of the map axis in figure coordinates
pos_map = ax_map.get_position()   # Bbox: [x0, y0, x1, y1]
pos_ax1 = ax1.get_position()

# make the time-series panel share the same top/bottom as the map panel
new_pos_ax1 = [
    pos_ax1.x0,        # keep its left x
    pos_map.y0,        # use map's bottom
    pos_ax1.width,     # keep its width
    pos_map.height,    # use map's height
]
ax1.set_position(new_pos_ax1)
ax2.set_position(new_pos_ax1)  # twin y-axis has to match too

plt.show()


## Load Scalers

In [ ]:
scaler_dep = joblib.load(str(scalers_dir)+'/ae_scaler_S_dep.save')
scaler_vx = joblib.load(str(scalers_dir)+'/ae_scaler_S_vx.save')
scaler_vy = joblib.load(str(scalers_dir)+'/ae_scaler_S_vy.save')

t_scaler_dep = joblib.load(str(scalers_dir)+'/t_scaler_S_dep.save') 
b_bc1_scaler_dep = joblib.load(str(scalers_dir)+'/b_bc1_scaler_S_dep.save') 
b_bc2_scaler_dep = joblib.load(str(scalers_dir)+'/b_bc2_scaler_S_dep.save') 
ls_scaler_dep = joblib.load(str(scalers_dir)+'/ls_scaler_S_dep.save') 

t_scaler_vx = joblib.load(str(scalers_dir)+'/t_scaler_S_vx.save') 
b_bc1_scaler_vx = joblib.load(str(scalers_dir)+'/b_bc1_scaler_S_vx.save') 
b_bc2_scaler_vx = joblib.load(str(scalers_dir)+'/b_bc2_scaler_S_vx.save') 
ls_scaler_vx = joblib.load(str(scalers_dir)+'/ls_scaler_S_vx.save')

t_scaler_vy = joblib.load(str(scalers_dir)+'/t_scaler_S_vy.save') 
b_bc1_scaler_vy = joblib.load(str(scalers_dir)+'/b_bc1_scaler_S_vy.save') 
b_bc2_scaler_vy = joblib.load(str(scalers_dir)+'/b_bc2_scaler_S_vy.save') 
ls_scaler_vy = joblib.load(str(scalers_dir)+'/ls_scaler_S_vy.save') 


## Load test data

In [ ]:
time = dl.load_variables_adh(data_dir/'Inset_T5.198e+06_mn0.026250.npz', comp_names=['S_dep'], t_start = 480)['time'][::t_skip].copy() ## Ignoring the first five days of simulation data
Np = len(param_list_test)
Nn = mesh['nodes'].shape[0]
Ne = mesh['triangles'].shape[0]
Nt = time.shape[0]

dataset_dep = np.zeros((Np, Nt, Nn))  ## Augment snapshots with parameter value, if needed
dataset_vx = np.zeros((Np, Nt, Nn))  ## Augment snapshots with parameter value, if needed
dataset_vy = np.zeros((Np, Nt, Nn))  ## Augment snapshots with parameter value, if needed

for inx, param in tqdm(enumerate(param_list_test)):
    
    fname = f'Inset_T5.198e+06_mn{param:.6f}.npz'
    snap_dep = dl.load_variables_adh(data_dir/fname, comp_names=['S_dep'], t_start = 480)['S_dep'].T[::t_skip] ## Ignoring the first five days of simulation data
    snap_vx = dl.load_variables_adh(data_dir/fname, comp_names=['S_vx'], t_start = 480)['S_vx'].T[::t_skip] ## Ignoring the first five days of simulation data
    snap_vy = dl.load_variables_adh(data_dir/fname, comp_names=['S_vy'], t_start = 480)['S_vy'].T[::t_skip] ## Ignoring the first five days of simulation data
    
    dataset_dep[inx, :, :] = snap_dep
    dataset_vx[inx, :, :] = snap_vx
    dataset_vy[inx, :, :] = snap_vy
    


In [ ]:
snaps = [np.where(time/60/60/24==test_days[0])[0][0], np.where(time/60/60/24==test_days[1])[0][0]]
Nt_snaps = time[snaps[0]:snaps[1]].shape[0]
    
dataset_vx_resh = np.reshape(dataset_vx,(Np,Nt,Nn))
dataset_vx_crop = dataset_vx_resh[:,snaps[0]:snaps[1],:]
dataset_vx = np.reshape(dataset_vx_crop,(Np*Nt_snaps,Nn))

dataset_dep_resh = np.reshape(dataset_dep,(Np,Nt,Nn))
dataset_dep_crop = dataset_dep_resh[:,snaps[0]:snaps[1],:]
dataset_dep = np.reshape(dataset_dep_crop,(Np*Nt_snaps,Nn))

dataset_vy_resh = np.reshape(dataset_vy,(Np,Nt,Nn))
dataset_vy_crop = dataset_vy_resh[:,snaps[0]:snaps[1],:]
dataset_vy = np.reshape(dataset_vy_crop,(Np*Nt_snaps,Nn))

if scaling:
    data_scaled_dep = scaler_dep.transform(dataset_dep)
    data_scaled_vx = scaler_vx.transform(dataset_vx)
    data_scaled_vy = scaler_vy.transform(dataset_vy)

dataset_r_dep = np.reshape(dataset_dep,(Np,Nt_snaps,Nn))
dataset_r_vx = np.reshape(dataset_vx,(Np,Nt_snaps,Nn))
dataset_r_vy = np.reshape(dataset_vy,(Np,Nt_snaps,Nn))


In [ ]:
data_ls_dep = ae_model_dep.encoder(data_scaled_dep)
latent_dim_dep = data_ls_dep.shape[1]
data_ls_dep = np.array(np.reshape(data_ls_dep,(Np,Nt_snaps,latent_dim_dep)))

data_ls_vx = ae_model_vx.encoder(data_scaled_vx)
latent_dim_vx = data_ls_vx.shape[1]
data_ls_vx = np.array(np.reshape(data_ls_vx,(Np,Nt_snaps,latent_dim_vx)))

data_ls_vy = ae_model_vy.encoder(data_scaled_vy)
latent_dim_vy = data_ls_vy.shape[1]
data_ls_vy = np.array(np.reshape(data_ls_vy,(Np,Nt_snaps,latent_dim_vy)))

In [ ]:
d_nodes = sett.discharge_nodes
t_nodes = sett.tailwater_nodes
b_par_data_dep, b_ic_data_dep, b_bc1_data_dep, b_bc2_data_dep, t_data_dep, target_data_dep = pu.multiple_param(param_list_test, data_ls_dep, 
                                                                                              time, snaps[0], snaps[1], 
                                                                                              t_skip, data_dir, d_nodes, t_nodes)
b_par_data_vx, b_ic_data_vx, b_bc1_data_vx, b_bc2_data_vx, t_data_vx, target_data_vx = pu.multiple_param(param_list_test, data_ls_vx, 
                                                                                         time, snaps[0], snaps[1], 
                                                                                         t_skip, data_dir, d_nodes, t_nodes)
b_par_data_vy, b_ic_data_vy, b_bc1_data_vy, b_bc2_data_vy, t_data_vy, target_data_vy = pu.multiple_param(param_list_test, data_ls_vy, 
                                                                                         time, snaps[0], snaps[1], 
                                                                                         t_skip, data_dir, d_nodes, t_nodes)

In [ ]:
if scaling is True:
    b_bc1_data_dep = b_bc1_scaler_dep.transform(b_bc1_data_dep)
    b_bc2_data_dep = b_bc2_scaler_dep.transform(b_bc2_data_dep)
    b_ic_data_dep = ls_scaler_dep.transform(b_ic_data_dep)
    target_data_dep = ls_scaler_dep.transform(target_data_dep)

    b_bc1_data_vx = b_bc1_scaler_vx.transform(b_bc1_data_vx)
    b_bc2_data_vx = b_bc2_scaler_vx.transform(b_bc2_data_vx)
    b_ic_data_vx = ls_scaler_vx.transform(b_ic_data_vx)
    target_data_vx = ls_scaler_vx.transform(target_data_vx)

    b_bc1_data_vy = b_bc1_scaler_vy.transform(b_bc1_data_vy)
    b_bc2_data_vy = b_bc2_scaler_vy.transform(b_bc2_data_vy)
    b_ic_data_vy = ls_scaler_vy.transform(b_ic_data_vy)
    target_data_vy = ls_scaler_vy.transform(target_data_vy)

In [ ]:
b_par_data_dep = np.reshape(b_par_data_dep,(Np,Nt_snaps,1))
b_ic_data_dep = np.reshape(b_ic_data_dep,(Np,Nt_snaps,latent_dim_dep))
b_bc1_data_dep = np.reshape(b_bc1_data_dep,(Np,Nt_snaps,len(d_nodes)))
b_bc2_data_dep = np.reshape(b_bc2_data_dep,(Np,Nt_snaps,len(t_nodes)))
t_data_dep = np.array((0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.))

b_par_data_vx = np.reshape(b_par_data_vx,(Np,Nt_snaps,1))
b_ic_data_vx = np.reshape(b_ic_data_vx,(Np,Nt_snaps,latent_dim_vx))
b_bc1_data_vx = np.reshape(b_bc1_data_vx,(Np,Nt_snaps,len(d_nodes)))
b_bc2_data_vx = np.reshape(b_bc2_data_vx,(Np,Nt_snaps,len(t_nodes)))
t_data_vx = np.array((0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.))
# t_data_vx = np.array((0.20, 0.40, 0.60, 0.80, 1.))

b_par_data_vy = np.reshape(b_par_data_vy,(Np,Nt_snaps,1))
b_ic_data_vy = np.reshape(b_ic_data_vy,(Np,Nt_snaps,latent_dim_vy))
b_bc1_data_vy = np.reshape(b_bc1_data_vy,(Np,Nt_snaps,len(d_nodes)))
b_bc2_data_vy = np.reshape(b_bc2_data_vy,(Np,Nt_snaps,len(t_nodes)))
t_data_vy = np.array((0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.))
# t_data_vy = np.array((0.20, 0.40, 0.60, 0.80, 1.))


### Predictions 1

These functions need fixing, doesnt work well if the dataset is not a factor of the window_size

In [ ]:
window_size = 10
res_dep = np.zeros((Np, Nt_snaps, latent_dim_dep))

for p in tqdm(range(Np)):
    b_ic = b_ic_data_dep[p, 0, :]
    res_dep[p,0,:] = b_ic
    for i in range(0, Nt_snaps, window_size):
        if i+window_size < Nt_snaps:
            par_input = b_par_data_dep[p, i+1:i+window_size+1, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
            bc1_input = b_bc1_data_dep[p, i+1:i+window_size+1, :] 
            bc2_input = b_bc2_data_dep[p, i+1:i+window_size+1, :] 
            t_input = np.expand_dims(t_data_dep[0:window_size],1)  
            prediction = mito_model_dep([par_input, ic_input, bc1_input, bc2_input, t_input])
            res_dep[p,i+1:i+window_size+1] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
        else:
            diff = Nt_snaps - i - 1
            par_input = b_par_data_dep[p, i+1:i+diff+1, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
            bc1_input = b_bc1_data_dep[p, i+1:i+diff+1, :] 
            bc2_input = b_bc2_data_dep[p, i+1:i+diff+1, :] 
            t_input = np.expand_dims(t_data_dep[0:diff],1)  

            prediction = mito_model_dep([par_input, ic_input, bc1_input, bc2_input, t_input])
            res_dep[p,i+1:i+diff+1] = prediction

res_dep = ls_scaler_dep.inverse_transform(res_dep.reshape(Np*Nt_snaps,latent_dim_dep))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_dep = ae_model_dep.decoder(res_dep).numpy()
full_res_dep = np.squeeze(scaler_dep.inverse_transform(full_res_dep))

# Reshape data for analysis and visualization
full_res_r_dep = np.reshape(full_res_dep, (Np, Nt_snaps, Nn))


In [ ]:
window_size = 10
res_vx = np.zeros((Np, Nt_snaps, latent_dim_vx))

for p in tqdm(range(Np)):
    b_ic = b_ic_data_vx[p, 0, :]
    res_vx[p,0,:] = b_ic
    for i in range(0, Nt_snaps, window_size):
        if i+window_size < Nt_snaps:
            par_input = b_par_data_vx[p, i+1:i+window_size+1, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
            bc1_input = b_bc1_data_vx[p, i+1:i+window_size+1, :] 
            bc2_input = b_bc2_data_vx[p, i+1:i+window_size+1, :] 
            t_input = np.expand_dims(t_data_vx[0:window_size],1)  
            prediction = mito_model_vx([par_input, ic_input, bc1_input, bc2_input, t_input])
            res_vx[p,i+1:i+window_size+1] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
        else:
            diff = Nt_snaps - i - 1
            par_input = b_par_data_vx[p, i+1:i+diff+1, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
            bc1_input = b_bc1_data_vx[p, i+1:i+diff+1, :] 
            bc2_input = b_bc2_data_vx[p, i+1:i+diff+1, :] 
            t_input = np.expand_dims(t_data_vx[0:diff],1)  

            prediction = mito_model_vx([par_input, ic_input, bc1_input, bc2_input, t_input])
            res_vx[p,i+1:i+diff+1] = prediction

res_vx = ls_scaler_vx.inverse_transform(res_vx.reshape(Np*Nt_snaps,latent_dim_vx))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_vx = ae_model_vx.decoder(res_vx).numpy()
full_res_vx = np.squeeze(scaler_vx.inverse_transform(full_res_vx))

# Reshape data for analysis and visualization
full_res_r_vx = np.reshape(full_res_vx, (Np, Nt_snaps, Nn))



In [ ]:
window_size = 10
res_vy = np.zeros((Np, Nt_snaps, latent_dim_vy))

for p in tqdm(range(Np)):
    b_ic = b_ic_data_vy[p, 0, :]
    res_vy[p,0,:] = b_ic
    for i in range(0, Nt_snaps, window_size):
        if i+window_size < Nt_snaps:
            par_input = b_par_data_vy[p, i+1:i+window_size+1, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
            bc1_input = b_bc1_data_vy[p, i+1:i+window_size+1, :] 
            bc2_input = b_bc2_data_vy[p, i+1:i+window_size+1, :] 
            t_input = np.expand_dims(t_data_vy[0:window_size],1)  
            prediction = mito_model_vy([par_input, ic_input, bc1_input, bc2_input, t_input])
            res_vy[p,i+1:i+window_size+1] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
        else:
            diff = Nt_snaps - i - 1
            par_input = b_par_data_vy[p, i+1:i+diff+1, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
            bc1_input = b_bc1_data_vy[p, i+1:i+diff+1, :] 
            bc2_input = b_bc2_data_vy[p, i+1:i+diff+1, :] 
            t_input = np.expand_dims(t_data_vy[0:diff],1)  

            prediction = mito_model_vy([par_input, ic_input, bc1_input, bc2_input, t_input])
            res_vy[p,i+1:i+diff+1] = prediction

res_vy = ls_scaler_vy.inverse_transform(res_vy.reshape(Np*Nt_snaps,latent_dim_vy))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_vy = ae_model_vy.decoder(res_vy).numpy()
full_res_vy = np.squeeze(scaler_vy.inverse_transform(full_res_vy))

# Reshape data for analysis and visualization
full_res_r_vy = np.reshape(full_res_vy, (Np, Nt_snaps, Nn))


#### Plot1

In [ ]:
mask_num = 0
rmse_dep = np.zeros((Np,Nt_snaps))
rmse_vx = np.zeros((Np,Nt_snaps))
rmse_vy = np.zeros((Np,Nt_snaps))

max_array_dep = np.zeros(Np)
max_array_vx = np.zeros(Np)
max_array_vy = np.zeros(Np)

n_rmse_dep = np.zeros((Np,Nt_snaps))
n_rmse_vx = np.zeros((Np,Nt_snaps))
n_rmse_vy = np.zeros((Np,Nt_snaps))


for p, param in enumerate(param_list_test[0:]):
    rmse_dep[p] = np.sqrt(np.mean((full_res_r_dep[p, :, mask_num:] - dataset_r_dep[p, :, mask_num:]) ** 2, axis=1)) 
    rmse_vx[p] = np.sqrt(np.mean((full_res_r_vx[p, :, mask_num:] - dataset_r_vx[p, :, mask_num:]) ** 2, axis=1)) 
    rmse_vy[p] = np.sqrt(np.mean((full_res_r_vy[p, :, mask_num:] - dataset_r_vy[p, :, mask_num:]) ** 2, axis=1)) 
    
    max_array_dep[p] = (np.abs(np.max(dataset_r_dep[p, :, mask_num:])-np.min(dataset_r_dep[p,  :, mask_num:])))
    max_array_vx[p] = (np.abs(np.max(dataset_r_vx[p, :, mask_num:])-np.min(dataset_r_vx[p,  :, mask_num:])))
    max_array_vy[p] = (np.abs(np.max(dataset_r_vy[p, :, mask_num:])-np.min(dataset_r_vy[p,  :, mask_num:])))

    n_rmse_dep[p] = rmse_dep[p]/max_array_dep[p]
    n_rmse_vx[p] = rmse_vx[p]/max_array_vx[p]
    n_rmse_vy[p] = rmse_vy[p]/max_array_vy[p]


In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(15,11), layout='constrained', sharey='row')

# Boxplot for Water Surface Elevation
axs[0].boxplot(rmse_dep.T, tick_labels=param_list_test)
axs[0].set_ylabel('Water Surface Elevation  (m)')

# Boxplot for U-Velocity
axs[1].boxplot(rmse_vx.T, tick_labels=param_list_test)
axs[1].set_ylabel('U-Velocity   (m/s)')

# Boxplot for V-Velocity
axs[2].boxplot(rmse_vy.T, tick_labels=param_list_test)
axs[2].set_ylabel('V-Velocity  (m/s)')
axs[2].set_xlabel('Roughness Coefficients', fontsize=16)
axs[2].tick_params(axis='x', which='major', rotation=30, labelsize=12)

axs[0].xaxis.set_tick_params(labelbottom=False)
axs[1].xaxis.set_tick_params(labelbottom=False)

# Calculate the min and max values for each dataset
min_dep, max_dep = np.min(dataset_r_dep), np.max(dataset_r_dep)
min_vx, max_vx = np.min(dataset_r_vx), np.max(dataset_r_vx)
min_vy, max_vy = np.min(dataset_r_vy), np.max(dataset_r_vy)

# Add a text box showing the extent for H, U, and V
extent_text_dep = f'{min_dep:.2f} < H < {max_dep:.2f}'
extent_text_vx = f'{min_vx:.2f} < U < {max_vx:.2f}'
extent_text_vy = f'{min_vy:.2f} < V < {max_vy:.2f}'

# Add text box for Water Surface Elevation
axs[0].text(0.16, 0.95, extent_text_dep, transform=axs[0].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text box for U-Velocity
axs[1].text(0.98, 0.95, extent_text_vx, transform=axs[1].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text box for V-Velocity
axs[2].text(0.4, 0.95, extent_text_vy, transform=axs[2].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))



In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(15,11), layout='constrained', sharey='row')

# violin for Water Surface Elevation
axs[0].violinplot(rmse_dep.T,showmeans=True)
axs[0].set_ylabel('Water Surface Elevation  (m)')

# violin for U-Velocity
axs[1].violinplot(rmse_vx.T,showmeans=True)
axs[1].set_ylabel('U-Velocity   (m/s)')

# violin for V-Velocity
axs[2].violinplot(rmse_vy.T,showmeans=True)
axs[2].set_ylabel('V-Velocity  (m/s)')
axs[2].set_xlabel('Roughness Coefficients', fontsize=16)
axs[2].tick_params(axis='x', which='major', rotation=30, labelsize=12)

axs[0].xaxis.set_tick_params(labelbottom=False)
axs[1].xaxis.set_tick_params(labelbottom=False)

# Calculate the min and max values for each dataset
min_dep, max_dep = np.min(dataset_r_dep), np.max(dataset_r_dep)
min_vx, max_vx = np.min(dataset_r_vx), np.max(dataset_r_vx)
min_vy, max_vy = np.min(dataset_r_vy), np.max(dataset_r_vy)

# Add a text box showing the extent for H, U, and V
extent_text_dep = f'{min_dep:.2f} < H < {max_dep:.2f}'
extent_text_vx = f'{min_vx:.2f} < U < {max_vx:.2f}'
extent_text_vy = f'{min_vy:.2f} < V < {max_vy:.2f}'

# Add text box for Water Surface Elevation
axs[0].text(0.16, 0.95, extent_text_dep, transform=axs[0].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text box for U-Velocity
axs[1].text(0.98, 0.95, extent_text_vx, transform=axs[1].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text box for V-Velocity
axs[2].text(0.4, 0.95, extent_text_vy, transform=axs[2].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))


#### Plot2

In [ ]:
train_snaps1 = [np.where(time/60/60/24==train_days0[0])[0][0], np.where(time/60/60/24==train_days1[0])[0][0]]
train_snaps2 = [np.where(time/60/60/24==train_days0[1])[0][0], np.where(time/60/60/24==train_days1[1])[0][0]]
train_snaps3 = [np.where(time/60/60/24==train_days0[2])[0][0], np.where(time/60/60/24==train_days1[2])[0][0]]


In [ ]:
x_train_lower1 = time[train_snaps1[0]]/3600/24
x_train_lower2 = time[train_snaps2[0]]/3600/24
x_train_lower3 = time[train_snaps3[0]]/3600/24

x_train_upper1 = time[train_snaps1[1]]/3600/24
x_train_upper2 = time[train_snaps2[1]]/3600/24
x_train_upper3 = time[train_snaps3[1]]/3600/24

x_lower = time[snaps[0]]/3600/24 
x_upper = time[snaps[1]]/3600/24
time2 = time[snaps[0]:snaps[1]]/60/60/24

colormap = [cmocean.cm.balance(0.375), cmocean.cm.balance(0.25), cmocean.cm.balance(0.125)]

z_order=[2,0,1]

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(15,11), layout='constrained', sharex=True)

count = 0

for p in [0, 1, -1]:

        axs[1].plot(time2,rmse_vx[p],label='r: '+str(param_list_test[p]), linewidth=3, linestyle='-', color=colormap[count], zorder=z_order[count])
        axs[1].set_xlim(x_lower,x_upper)


        axs[2].plot(time2,rmse_vy[p],label='r: '+str(param_list_test[p]), linewidth=3, linestyle='-', color=colormap[count], zorder=z_order[count])
        axs[2].set_xlim(x_lower,x_upper)
        axs[2].set_xlabel('Time (days)')
        
        axs[0].plot(time2,rmse_dep[p],label='r: '+str(param_list_test[p]), linewidth=3, linestyle='-', color=colormap[count], zorder=z_order[count])
        axs[0].set_xlim(x_lower,x_upper)
        axs[0].legend(bbox_to_anchor=(0.5,1.15), ncol=3, loc='center', title='Bottom Friction Coefficients', title_fontsize=15, fontsize=12.75)

        count += 1

axs[0].axvline(x_train_upper1,color='black',linestyle="--")
axs[0].axvline(x_train_lower1,color='black',linestyle="--")
axs[1].axvline(x_train_upper1,color='black',linestyle="--")
axs[1].axvline(x_train_lower1,color='black',linestyle="--")
axs[2].axvline(x_train_upper1,color='black',linestyle="--")
axs[2].axvline(x_train_lower1,color='black',linestyle="--")

axs[0].axvline(x_train_upper2,color='black',linestyle="--")
axs[0].axvline(x_train_lower2,color='black',linestyle="--")
axs[1].axvline(x_train_upper2,color='black',linestyle="--")
axs[1].axvline(x_train_lower2,color='black',linestyle="--")
axs[2].axvline(x_train_upper2,color='black',linestyle="--")
axs[2].axvline(x_train_lower2,color='black',linestyle="--")

axs[0].axvline(x_train_upper3,color='black',linestyle="--")
axs[0].axvline(x_train_lower3,color='black',linestyle="--")
axs[1].axvline(x_train_upper3,color='black',linestyle="--")
axs[1].axvline(x_train_lower3,color='black',linestyle="--")
axs[2].axvline(x_train_upper3,color='black',linestyle="--")
axs[2].axvline(x_train_lower3,color='black',linestyle="--")

axs[0].axvspan(x_train_lower1, x_train_upper1, color='grey', alpha=0.10, zorder=-1)
axs[0].axvspan(x_train_lower2, x_train_upper2, color='grey', alpha=0.10, zorder=-1)
axs[0].axvspan(x_train_lower3, x_train_upper3, color='grey', alpha=0.10, zorder=-1)

axs[1].axvspan(x_train_lower1, x_train_upper1, color='grey', alpha=0.10, zorder=-1)
axs[1].axvspan(x_train_lower2, x_train_upper2, color='grey', alpha=0.10, zorder=-1)
axs[1].axvspan(x_train_lower3, x_train_upper3, color='grey', alpha=0.10, zorder=-1)

axs[2].axvspan(x_train_lower1, x_train_upper1, color='grey', alpha=0.10, zorder=-1)
axs[2].axvspan(x_train_lower2, x_train_upper2, color='grey', alpha=0.10, zorder=-1)
axs[2].axvspan(x_train_lower3, x_train_upper3, color='grey', alpha=0.10, zorder=-1)


axs[0].set_ylabel('Water Surface Elevation  (m)')

axs[1].set_ylabel('U-Velocity   (m/s)')

# violin for V-Velocity
axs[2].set_ylabel('V-Velocity  (m/s)')

#### Plot3

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(15,10), layout='constrained', gridspec_kw={'width_ratios': [1, 1, 1]})

axs[0,0].set_ylabel('Water Surface Elevation  (m)')
axs[1,0].set_ylabel('U-Velocity   (m/s)')
axs[2,0].set_ylabel('V-Velocity  (m/s)')

axs[0,2].set_ylabel('r: 0.02625', rotation=270, labelpad=20)
axs[1,2].set_ylabel('r: 0.02375', rotation=270, labelpad=20)
axs[2,2].set_ylabel('r: 0.02375', rotation=270, labelpad=20)
axs[0,2].yaxis.set_label_position("right")
axs[1,2].yaxis.set_label_position("right")
axs[2,2].yaxis.set_label_position("right")

axs[0,0].xaxis.set_tick_params(labelbottom=False)
axs[1,0].xaxis.set_tick_params(labelbottom=False)

axs[0,0].plot(time2,full_res_r_dep[-1,:,sensors[0]],label='_nolegend_', linewidth=3, linestyle='-',color=s1_color)
first = axs[0,0].plot(time2,dataset_r_dep[-1,:,sensors[0]],label='AdH', linewidth=2, linestyle='--',color='black')[0]
axs[0,1].plot(time2,full_res_r_dep[-1,:,sensors[1]],label='_nolegend_', linewidth=3, linestyle='-',color=s2_color)
axs[0,1].plot(time2,dataset_r_dep[-1,:,sensors[1]],label='_nolegend_', linewidth=2, linestyle='--',color='black')
axs[0,2].plot(time2,full_res_r_dep[-1,:,sensors[2]],label='_nolegend_', linewidth=3, linestyle='-',color=s3_color)
axs[0,2].plot(time2,dataset_r_dep[-1,:,sensors[2]],label='_nolegend_', linewidth=2, linestyle='--',color='black')
second = axs[1,0].plot(time2,full_res_r_vx[0,:,sensors[0]],label='MITONet (sensor 1)', linewidth=3, linestyle='-',color=s1_color)[0]
axs[1,0].plot(time2,dataset_r_vx[0,:,sensors[0]],label='_nolegend_', linewidth=2, linestyle='--',color='black')
third = axs[1,1].plot(time2,full_res_r_vx[0,:,sensors[1]],label='MITONet (sensor 2)', linewidth=3, linestyle='-',color=s2_color)[0]
axs[1,1].plot(time2,dataset_r_vx[0,:,sensors[1]],label='_nolegend_', linewidth=2, linestyle='--',color='black')
fourth = axs[1,2].plot(time2,full_res_r_vx[0,:,sensors[2]],label='MITONet (sensor 3)', linewidth=3, linestyle='-',color=s3_color)[0]
axs[1,2].plot(time2,dataset_r_vx[0,:,sensors[2]],label='_nolegend_', linewidth=2, linestyle='--',color='black')
axs[2,0].plot(time2,full_res_r_vy[0,:,sensors[0]],label='_nolegend_', linewidth=3, linestyle='-',color=s1_color)
axs[2,0].plot(time2,dataset_r_vy[0,:,sensors[0]],label='_nolegend_', linewidth=2, linestyle='--',color='black')
axs[2,1].plot(time2,full_res_r_vy[0,:,sensors[1]],label='_nolegend_', linewidth=3, linestyle='-',color=s2_color)
axs[2,1].plot(time2,dataset_r_vy[0,:,sensors[1]],label='_nolegend_', linewidth=2, linestyle='--',color='black')
axs[2,2].plot(time2,full_res_r_vy[0,:,sensors[2]],label='_nolegend_', linewidth=3, linestyle='-',color=s3_color)
axs[2,2].plot(time2,dataset_r_vy[0,:,sensors[2]],label='_nolegend_', linewidth=2, linestyle='--',color='black')
plt.setp(axs[:,0:], xlim=[5,60])
# plt.setp(axs[0,:], ylim=[8,15])
fig.legend(bbox_to_anchor=(0.9, 1.05), handles=[first, second, third, fourth], labels=['AdH', 'MITONet (sensor1)', 'MITONet (sensor2)', 'MITONet (sensor3)'], ncol=4, fontsize=15)

axs[2,0].set_xlabel('Time (days)')
axs[2,1].set_xlabel('Time (days)')
axs[2,2].set_xlabel('Time (days)')

axs[0,0].xaxis.set_tick_params(labelbottom=False)
axs[1,0].xaxis.set_tick_params(labelbottom=False)
axs[0,1].xaxis.set_tick_params(labelbottom=False)
axs[1,1].xaxis.set_tick_params(labelbottom=False)
axs[0,2].xaxis.set_tick_params(labelbottom=False)
axs[1,2].xaxis.set_tick_params(labelbottom=False)

#### Plot4

In [ ]:
tstep = -1
param1 = 0
param2 = -1

fig, axs = plt.subplots(2, 3, figsize=(15, 8), layout='constrained')#, sharey='col')
# axs[0,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], full_res_r_dep[0,0,:], vmin=-0.5, vmax=0.5, cmap='cmo.balance', shading='gouraud')
axs[0,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                   dataset_r_dep[param2,tstep,:], vmin=dataset_r_dep[param2,tstep,:].min(), 
                   vmax=dataset_r_dep[param2,tstep,:].max(), cmap='cmo.speed', shading='gouraud')
axs[0,0].axis('off')
last_dep = axs[1,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                   full_res_r_dep[param2,tstep,:], vmin=dataset_r_dep[param2,tstep,:].min(), 
                   vmax=dataset_r_dep[param2,tstep,:].max(), cmap='cmo.speed', shading='gouraud')
axs[1,0].axis('off')

axs[0,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                   dataset_r_vx[param1,tstep,:], vmin=dataset_r_vx[param1,tstep,:].min(), 
                   vmax=dataset_r_vx[param1,tstep,:].max(), cmap='cmo.delta', shading='gouraud')
axs[0,1].axis('off')
last_vx = axs[1,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                   full_res_r_vx[param1,tstep,:], vmin=dataset_r_vx[param1,tstep,:].min(), 
                   vmax=dataset_r_vx[param1,tstep,:].max(), cmap='cmo.delta', shading='gouraud')
axs[1,1].axis('off')

axs[0,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                   dataset_r_vy[param1,tstep,:], vmin=dataset_r_vy[param1,tstep,:].min(), 
                   vmax=dataset_r_vy[param1,tstep,:].max(), cmap='cmo.delta', shading='gouraud')
axs[0,2].axis('off')
last_vy = axs[1,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                   full_res_r_vy[param1,tstep,:], vmin=dataset_r_vy[param1,tstep,:].min(), 
                   vmax=dataset_r_vy[param1,tstep,:].max(), cmap='cmo.delta', shading='gouraud')
axs[1,2].axis('off')

axs[0, 0].text(-0.05, 0.5, 'AdH', size=20, ha="center", va="center", 
               rotation=90, transform=axs[0,0].transAxes)
axs[1, 0].text(-0.05, 0.5, 'MITONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[1,0].transAxes)

axs[0,0].set_title('r: 0.02625')
axs[0,1].set_title('r: 0.02375')
axs[0,2].set_title('r: 0.02375')

cbar = plt.colorbar(last_dep, orientation='horizontal', aspect=18)
cbar.set_label(label='Water Surface Elevation  (m)', size=20) 
cbar = plt.colorbar(last_vx, orientation='horizontal', aspect=18)
cbar.set_label(label='U-Velocity  (m/s)', size=20) 
cbar = plt.colorbar(last_vy, orientation='horizontal', aspect=18)
cbar.set_label(label='V-Velocity  (m/s)',  size=20) 

#try adding 1e-6 for relative error plots
# try absolute error
# try rmse
# try both and send

In [ ]:
param1 = 0
param2 = -1

# Precompute the two error fields:
err_dep  = np.mean(np.abs(dataset_r_dep[param2, :, :]  - full_res_r_dep[param2, :, :]),axis=0)
err_vyx   = np.mean(np.abs(dataset_r_vx[param1, :, :]   - full_res_r_vx[param1, :, :]),axis=0)
err_vy   = np.mean(np.abs(dataset_r_vy[param1, :, :]   - full_res_r_vy[param1, :, :]),axis=0)

fig, axs = plt.subplots(1, 3, figsize=(15,5), layout='constrained')#, sharey='col')
# axs[0,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, full_res_r_dep[0,0,:], vmin=-0.5, vmax=0.5, cmap='cmo.balance', shading='gouraud')

last_eh1 = axs[0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                              err_dep, vmin=0, 
                              vmax=err_dep.max(), cmap='cmo.amp', shading='gouraud')
axs[0].axis('off')

last_eu1 = axs[1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                              err_vyx, vmin=0, 
                              vmax=err_vyx.max(), cmap='cmo.amp', shading='gouraud')
axs[1].axis('off')

last_ev1 = axs[2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                              err_vy, vmin=0, 
                              vmax=err_vy.max(), cmap='cmo.amp', shading='gouraud')
axs[2].axis('off')

axs[0].set_title('Water Surface Elevation \n(r: 0.02625)', size=20,transform=axs[0].transAxes)
axs[1].set_title('U-Velocity \n(r: 0.02375)', size=20,transform=axs[1].transAxes)
axs[2].set_title('V-Velocity \n(r: 0.02375)', size=20,transform=axs[2].transAxes)


cbar = plt.colorbar(last_eh1, orientation='horizontal', aspect=18)
cbar.set_label(label='Mean Abs. Error  (m)', size=18) 
cbar = plt.colorbar(last_eu1, orientation='horizontal', aspect=18)
cbar.set_label(label='Mean Abs. Error  (m/s)', size=18) 
cbar = plt.colorbar(last_ev1, orientation='horizontal', aspect=18)
cbar.set_label(label='Mean Abs. Error  (m/s)',  size=18) 

#### ACC and RMSE Table

In [ ]:
##### ACC
P = dataset_r_dep.shape[0]
MM = dataset_r_dep.shape[1]
N = dataset_r_dep.shape[2]

ACC_dep = np.zeros(dataset_r_dep.shape[0])

for p in range(P):
    num = np.zeros(MM)
    den = np.zeros(MM)
    for j in range(MM):
        Pj = np.sum(full_res_r_dep[p,j,:])/N
        Tj = np.sum(dataset_r_dep[p,j,:])/N
        num[j] = np.sum((dataset_r_dep[p,j,:]-Tj)*(full_res_r_dep[p,j,:]-Pj))
        den[j] = np.sqrt(np.sum((dataset_r_dep[p,j,:]-Tj)**2)*np.sum((full_res_r_dep[p,j,:]-Pj)**2))
    ACC_dep[p] = np.sum(num/den)/MM
        # for i in range(dataset_r_dep.shape[2]):
            
print('ACC:'+str(list(ACC_dep)))
print('RMSE:'+str(list(np.mean(rmse_dep,axis=1))))
print('nRMSE:'+str(list(np.mean(n_rmse_dep,axis=1))))
print('uRMSE:'+str(list(np.mean(ub_s_rmse_dep,axis=1))))
print('Max RMSE:'+str(list(np.max(rmse_dep,axis=1))))


In [ ]:
##### ACC

P = dataset_r_vx.shape[0]
MM = dataset_r_vx.shape[1]
N = dataset_r_vx.shape[2]

ACC_vx = np.zeros(dataset_r_vx.shape[0])

for p in range(P):
    num = np.zeros(MM)
    den = np.zeros(MM)
    for j in range(MM):
        Pj = np.sum(full_res_r_vx[p,j,:])/N
        Tj = np.sum(dataset_r_vx[p,j,:])/N
        num[j] = np.sum((dataset_r_vx[p,j,:]-Tj)*(full_res_r_vx[p,j,:]-Pj))
        den[j] = np.sqrt(np.sum((dataset_r_vx[p,j,:]-Tj)**2)*np.sum((full_res_r_vx[p,j,:]-Pj)**2))
    ACC_vx[p] = np.sum(num/den)/MM
        # for i in range(dataset_r_vx.shape[2]):
            
print('ACC:'+str(list(ACC_vx)))
print('RMSE:'+str(list(np.mean(rmse_vx,axis=1))))
print('nRMSE:'+str(list(np.mean(n_rmse_vx,axis=1))))
print('uRMSE:'+str(list(np.mean(ub_s_rmse_vx,axis=1))))
print('Max RMSE:'+str(list(np.max(rmse_vx,axis=1))))


In [ ]:
##### ACC

P = dataset_r_vy.shape[0]
MM = dataset_r_vy.shape[1]
N = dataset_r_vy.shape[2]

ACC_vy = np.zeros(dataset_r_vy.shape[0])

for p in range(P):
    num = np.zeros(MM)
    den = np.zeros(MM)
    for j in range(MM):
        Pj = np.sum(full_res_r_vy[p,j,:])/N
        Tj = np.sum(dataset_r_vy[p,j,:])/N
        num[j] = np.sum((dataset_r_vy[p,j,:]-Tj)*(full_res_r_vy[p,j,:]-Pj))
        den[j] = np.sqrt(np.sum((dataset_r_vy[p,j,:]-Tj)**2)*np.sum((full_res_r_vy[p,j,:]-Pj)**2))
    ACC_vy[p] = np.sum(num/den)/MM
        # for i in range(dataset_r_vy.shape[2]):
            
print('ACC:'+str(list(ACC_vy)))
print('RMSE:'+str(list(np.mean(rmse_vy,axis=1))))
print('nRMSE:'+str(list(np.mean(n_rmse_vy,axis=1))))
print('uRMSE:'+str(list(np.mean(ub_s_rmse_vy,axis=1))))
print('Max RMSE:'+str(list(np.max(rmse_vy,axis=1))))


### Predictions 2

In [ ]:
window_size = 10
day_window = 5.
day_total = test_days[1]-test_days[0]

n_windows = int(day_total/day_window)

snap_windows = int(Nt_snaps/day_total*day_window)

rmse_array_dep = np.zeros((n_windows, len(param_list_test)))
rmse_array_vx = np.zeros((n_windows, len(param_list_test)))
rmse_array_vy = np.zeros((n_windows, len(param_list_test)))

max_array_dep = np.zeros((n_windows, len(param_list_test)))
max_array_vx = np.zeros((n_windows, len(param_list_test)))
max_array_vy = np.zeros((n_windows, len(param_list_test)))

In [ ]:
current_window = 0

for w in tqdm(range(n_windows)):
    res_dep = np.zeros((Np, snap_windows, latent_dim_dep))
    res_vx = np.zeros((Np, snap_windows, latent_dim_vx))
    res_vy = np.zeros((Np, snap_windows, latent_dim_vy))
    
    for p in range(Np):

        b_ic_dep = ls_scaler_dep.transform(np.expand_dims(data_ls_dep[p, current_window, :],0))[0]
        res_dep[p,0,:] = b_ic_dep
        
        b_ic_vx = ls_scaler_vx.transform(np.expand_dims(data_ls_vx[p, current_window, :],0))[0]
        res_vx[p,0,:] = b_ic_vx
        
        b_ic_vy = ls_scaler_vy.transform(np.expand_dims(data_ls_vy[p, current_window, :],0))[0]
        res_vy[p,0,:] = b_ic_vy
        
        for i in range(0, snap_windows, window_size):
            temp_i = i + current_window

            if i+window_size < snap_windows:
                
                par_input_dep = b_par_data_dep[p, temp_i+1:temp_i+window_size+1, :]  
                ic_input_dep = np.tile(np.expand_dims(b_ic_dep,1),window_size).T
                bc1_input_dep = b_bc1_data_dep[p, temp_i+1:temp_i+window_size+1, :] 
                bc2_input_dep = b_bc2_data_dep[p, temp_i+1:temp_i+window_size+1, :] 
                t_input_dep = np.expand_dims(t_data_dep[0:window_size],1)  
                prediction_dep = mito_model_dep([par_input_dep, ic_input_dep, bc1_input_dep, bc2_input_dep, t_input_dep])
                res_dep[p,i+1:i+window_size+1] = prediction_dep
                b_ic_dep = prediction_dep[-1]  # Update initial condition for the next iteration

                par_input_vx = b_par_data_vx[p, temp_i+1:temp_i+window_size+1, :]  
                ic_input_vx = np.tile(np.expand_dims(b_ic_vx,1),window_size).T
                bc1_input_vx = b_bc1_data_vx[p, temp_i+1:temp_i+window_size+1, :] 
                bc2_input_vx = b_bc2_data_vx[p, temp_i+1:temp_i+window_size+1, :] 
                t_input_vx = np.expand_dims(t_data_vx[0:window_size],1)  
                prediction_vx = mito_model_vx([par_input_vx, ic_input_vx, bc1_input_vx, bc2_input_vx, t_input_vx])
                res_vx[p,i+1:i+window_size+1] = prediction_vx
                b_ic_vx = prediction_vx[-1]  # Update initial condition for the next iteration

                par_input_vy = b_par_data_vy[p, temp_i+1:temp_i+window_size+1, :]  
                ic_input_vy = np.tile(np.expand_dims(b_ic_vy,1),window_size).T
                bc1_input_vy = b_bc1_data_vy[p, temp_i+1:temp_i+window_size+1, :] 
                bc2_input_vy = b_bc2_data_vy[p, temp_i+1:temp_i+window_size+1, :] 
                t_input_vy = np.expand_dims(t_data_vy[0:window_size],1)  
                prediction_vy = mito_model_vy([par_input_vy, ic_input_vy, bc1_input_vy, bc2_input_vy, t_input_vy])
                res_vy[p,i+1:i+window_size+1] = prediction_vy
                b_ic_vy = prediction_vy[-1]  # Update initial condition for the next iteration
            else:
                diff = snap_windows - i - 1
                
                par_input_dep = b_par_data_dep[p, temp_i+1:temp_i+1+diff, :]  
                ic_input_dep = np.tile(np.expand_dims(b_ic_dep,1),diff).T
                bc1_input_dep = b_bc1_data_dep[p, temp_i+1:temp_i+1+diff, :] 
                bc2_input_dep = b_bc2_data_dep[p, temp_i+1:temp_i+1+diff, :] 
                t_input_dep = np.expand_dims(t_data_dep[0:diff],1)  
                prediction_dep = mito_model_dep([par_input_dep, ic_input_dep, bc1_input_dep, bc2_input_dep, t_input_dep])
                res_dep[p,i+1:i+diff+1] = prediction_dep
                
                par_input_vx = b_par_data_vx[p, temp_i+1:temp_i+1+diff, :]  
                ic_input_vx = np.tile(np.expand_dims(b_ic_vx,1),diff).T
                bc1_input_vx = b_bc1_data_vx[p, temp_i+1:temp_i+1+diff, :] 
                bc2_input_vx = b_bc2_data_vx[p, temp_i+1:temp_i+1+diff, :] 
                t_input_vx = np.expand_dims(t_data_vx[0:diff],1)  
                prediction_vx = mito_model_vx([par_input_vx, ic_input_vx, bc1_input_vx, bc2_input_vx, t_input_vx])
                res_vx[p,i+1:i+diff+1] = prediction_vx
                
                par_input_vy = b_par_data_vy[p, temp_i+1:temp_i+1+diff, :]  
                ic_input_vy = np.tile(np.expand_dims(b_ic_vy,1),diff).T
                bc1_input_vy = b_bc1_data_vy[p, temp_i+1:temp_i+1+diff, :] 
                bc2_input_vy = b_bc2_data_vy[p, temp_i+1:temp_i+1+diff, :] 
                t_input_vy = np.expand_dims(t_data_vy[0:diff],1)  
                prediction_vy = mito_model_vy([par_input_vy, ic_input_vy, bc1_input_vy, bc2_input_vy, t_input_vy])
                res_vy[p,i+1:i+diff+1] = prediction_vy

    res_dep = ls_scaler_dep.inverse_transform(res_dep.reshape(Np*snap_windows,latent_dim_dep))
    # res_dep = ls_scaler_dep.inverse_transform(res_dep)        
    full_res_dep = ae_model_dep.decoder(res_dep).numpy()
    full_res_dep = np.squeeze(scaler_dep.inverse_transform(full_res_dep))
    full_res_r_dep = np.reshape(full_res_dep, (Np, snap_windows, Nn))

    res_vx = ls_scaler_vx.inverse_transform(res_vx.reshape(Np*snap_windows,latent_dim_vx))
    # res_vx = ls_scaler_vx.inverse_transform(res_vx)        
    full_res_vx = ae_model_vx.decoder(res_vx).numpy()
    full_res_vx = np.squeeze(scaler_vx.inverse_transform(full_res_vx))
    full_res_r_vx = np.reshape(full_res_vx, (Np, snap_windows, Nn))

    res_vy = ls_scaler_vy.inverse_transform(res_vy.reshape(Np*snap_windows,latent_dim_vy))
    # res_vy = ls_scaler_vy.inverse_transform(res_vy)        
    full_res_vy = ae_model_vy.decoder(res_vy).numpy()
    full_res_vy = np.squeeze(scaler_vy.inverse_transform(full_res_vy))
    full_res_r_vy = np.reshape(full_res_vy, (Np, snap_windows, Nn))

    
    for p in range(Np):

        rmse_array_dep[w, p] = np.mean(np.sqrt(np.mean((full_res_r_dep[p, :] - dataset_r_dep[p, current_window:snap_windows+current_window] ) ** 2, axis=1)))
        max_array_dep[w, p] = (np.abs(np.max(dataset_r_dep[p, current_window:snap_windows+current_window])-np.min(dataset_r_dep[p, current_window:snap_windows+current_window])))
        
        rmse_array_vx[w, p] = np.mean(np.sqrt(np.mean((full_res_r_vx[p, :] - dataset_r_vx[p, current_window:snap_windows+current_window] ) ** 2, axis=1)))
        max_array_vx[w, p] = (np.abs(np.max(dataset_r_vx[p, current_window:snap_windows+current_window])-np.min(dataset_r_vx[p, current_window:snap_windows+current_window])))
        
        rmse_array_vy[w, p] = np.mean(np.sqrt(np.mean((full_res_r_vy[p, :] - dataset_r_vy[p, current_window:snap_windows+current_window] ) ** 2, axis=1)))
        max_array_vy[w, p] = (np.abs(np.max(dataset_r_vy[p, current_window:snap_windows+current_window])-np.min(dataset_r_vy[p, current_window:snap_windows+current_window])))

    current_window += snap_windows



#### Plot5

In [ ]:
colormap=['darkorange', 'darkred', 'darkgreen']

days_list = ['5-10','10-15','15-20','20-25','25-30','30-35',
             '35-40','40-45','45-50','50-55','55-60']

# Create an array of positions for our x-axis
x = np.arange(len(days_list))

# Set the width of each bar
width = 0.25

fig, ax = plt.subplots(3,figsize=(15, 12), sharex=True, layout='constrained')

# We'll create one bar for each column in rmse_array_dep
num_columns = 3

for p,i in enumerate([0,1,-1]):
    # Offset each bar group by i*width
    offset = width * p
    
    # Plot the bars for column i
    rects = ax[0].bar(x + offset,
                   rmse_array_dep[:, i]/max_array_dep[:, i],
                   width,
                   color = colormap[p],
                   label=f'r = {param_list_test[i]}')
    
    # Plot the bars for column i
    rects = ax[1].bar(x + offset,
                   rmse_array_vx[:, i]/max_array_vx[:, i],
                   width,
                   color = colormap[p],
                   label=f'r = {param_list_test[i]}')
    
    # Plot the bars for column i
    rects = ax[2].bar(x + offset,
                   rmse_array_vy[:, i]/max_array_vy[:, i],
                   width,
                   color = colormap[p],
                   label=f'r = {param_list_test[i]}')
    
# ax[0].set_ylim(0,0.009)
# ax[1].set_ylim(0,0.016)
# ax[2].set_ylim(0,0.014)

    
ax[0].set_ylabel('Water Surface Elevation')
ax[1].set_ylabel('U-Velocity')
ax[2].set_ylabel('V-Velocity')


# Adjust the x-axis ticks to be centered under the group
# We'll shift by width*(num_columns-1)/2 to center the labels
ax[2].set_xticks(x + width * (num_columns - 1) / 2)
ax[2].set_xticklabels(days_list)

ax[2].set_xlabel('Days')

ax[0].legend(bbox_to_anchor=(0.5,1.15), ncol=3, loc='center', title='Roughness Coefficients', title_fontsize=15, fontsize=12.75)

# Calculate the min and max values for each dataset
min_dep, max_dep = np.min(rmse_array_dep), np.max(rmse_array_dep)
min_vx, max_vx = np.min(rmse_array_vx), np.max(rmse_array_vx)
min_vy, max_vy = np.min(rmse_array_vy), np.max(rmse_array_vy)

# Add a text box showing the extent for H, U, and V
extent_text_dep = f'{min_dep:.2f} < $\overline{{RMSE}}$  (m) < {max_dep:.2f}'
extent_text_vx = f'{min_vx:.2f} < $\overline{{RMSE}}$  (m/s) < {max_vx:.2f}'
extent_text_vy = f'{min_vy:.2f} < $\overline{{RMSE}}$  (m/s) < {max_vy:.2f}'

# Add text box for Water Surface Elevation
ax[0].text(0.98, 0.95, extent_text_dep, transform=ax[0].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text box for U-Velocity
ax[1].text(0.98, 0.95, extent_text_vx, transform=ax[1].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text box for V-Velocity
ax[2].text(0.98, 0.95, extent_text_vy, transform=ax[2].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

fmt = '%.3f' # Format you want the ticks, e.g. '40%'
yticks = mtick.FormatStrFormatter(fmt)
ax[0].yaxis.set_major_formatter(yticks)
ax[1].yaxis.set_major_formatter(yticks)
ax[2].yaxis.set_major_formatter(yticks)



plt.show()

### Predictions 3

In [ ]:
window_size = 1

In [ ]:
if scaling:
    zero_dep = scaler_dep.transform(np.zeros((1,dataset_dep.shape[1])))
    zero_vx = scaler_vx.transform(np.zeros((1,dataset_vx.shape[1])))
    zero_vy = scaler_vy.transform(np.zeros((1,dataset_vy.shape[1])))

    zero_ic_dep = ls_scaler_dep.transform(ae_model_dep.encoder(zero_dep))
    zero_ic_vx = ls_scaler_vx.transform(ae_model_vx.encoder(zero_vx))
    zero_ic_vy = ls_scaler_vy.transform(ae_model_vy.encoder(zero_vy))

In [ ]:
snaps_crop = [np.where(time/3600/24==30)[0][0], np.where(time/3600/24==45)[0][0]]
Nt_snaps_crop = time[snaps_crop[0]:snaps_crop[1]].shape[0]
snaps_crop -= snaps[0]


In [ ]:
res_dep = np.zeros((Np, Nt_snaps_crop, latent_dim_dep))

for p in tqdm(range(Np)):
    b_ic = zero_ic_dep[0] 
    res_dep[p,0,:] = b_ic
    for i in range(snaps_crop[0],snaps_crop[1], window_size):
        if i+window_size < Nt_snaps:
            par_input = b_par_data_dep[p, i+1:i+window_size+1, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
            bc1_input = b_bc1_data_dep[p, i+1:i+window_size+1, :] 
            bc2_input = b_bc2_data_dep[p, i+1:i+window_size+1, :] 
            t_input = np.expand_dims(t_data_dep[0:window_size],1)  
    
            prediction = mito_model_dep([par_input, ic_input, bc1_input, bc2_input, t_input])
            res_dep[p,i+1-snaps_crop[0]:i+window_size+1-snaps_crop[0]] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
        else:
            diff = Nt_snaps - i - 1
            par_input = b_par_data_dep[p, i+1:i+1+diff, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
            bc1_input = b_bc1_data_dep[p, i+1:i+1+diff, :] 
            bc2_input = b_bc2_data_dep[p, i+1:i+1+diff, :] 
            t_input = np.expand_dims(t_data_dep[0:diff],1)  

            prediction = mito_model_dep([par_input, ic_input, bc1_input, bc2_input, t_input])
            res_dep[p,i+1-snaps_crop[0]:i+diff+1-snaps_crop[0]] = prediction
            

res_dep = ls_scaler_dep.inverse_transform(res_dep.reshape(Np*Nt_snaps_crop,latent_dim_dep))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_dep = ae_model_dep.decoder(res_dep).numpy()
full_res_dep = np.squeeze(scaler_dep.inverse_transform(full_res_dep))

# Reshape data for analysis and visualization
full_res_r_dep = np.reshape(full_res_dep, (Np, Nt_snaps_crop, Nn))


In [ ]:
res_vx = np.zeros((Np, Nt_snaps_crop, latent_dim_vx))

for p in tqdm(range(Np)):
    b_ic = zero_ic_vx[0] 
    res_vx[p,0,:] = b_ic
    for i in range(snaps_crop[0],snaps_crop[1], window_size):
        if i+window_size < Nt_snaps:
            par_input = b_par_data_vx[p, i+1:i+window_size+1, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
            bc1_input = b_bc1_data_vx[p, i+1:i+window_size+1, :] 
            bc2_input = b_bc2_data_vx[p, i+1:i+window_size+1, :] 
            t_input = np.expand_dims(t_data_vx[0:window_size],1)  
    
            prediction = mito_model_vx([par_input, ic_input, bc1_input, bc2_input, t_input])
            res_vx[p,i+1-snaps_crop[0]:i+window_size+1-snaps_crop[0]] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
        else:
            diff = Nt_snaps - i - 1
            par_input = b_par_data_vx[p, i+1:i+1+diff, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
            bc1_input = b_bc1_data_vx[p, i+1:i+1+diff, :] 
            bc2_input = b_bc2_data_vx[p, i+1:i+1+diff, :] 
            t_input = np.expand_dims(t_data_vx[0:diff],1)  

            prediction = mito_model_vx([par_input, ic_input, bc1_input, bc2_input, t_input])
            res_vx[p,i+1-snaps_crop[0]:i+diff+1-snaps_crop[0]] = prediction
            

res_vx = ls_scaler_vx.inverse_transform(res_vx.reshape(Np*Nt_snaps_crop,latent_dim_vx))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_vx = ae_model_vx.decoder(res_vx).numpy()
full_res_vx = np.squeeze(scaler_vx.inverse_transform(full_res_vx))

# Reshape data for analysis and visualization
full_res_r_vx = np.reshape(full_res_vx, (Np, Nt_snaps_crop, Nn))


In [ ]:
res_vy = np.zeros((Np, Nt_snaps_crop, latent_dim_vy))

for p in tqdm(range(Np)):
    b_ic = zero_ic_vy[0] 
    res_vy[p,0,:] = b_ic
    for i in range(snaps_crop[0],snaps_crop[1], window_size):
        if i+window_size < Nt_snaps:
            par_input = b_par_data_vy[p, i+1:i+window_size+1, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
            bc1_input = b_bc1_data_vy[p, i+1:i+window_size+1, :] 
            bc2_input = b_bc2_data_vy[p, i+1:i+window_size+1, :] 
            t_input = np.expand_dims(t_data_vy[0:window_size],1)  
    
            prediction = mito_model_vy([par_input, ic_input, bc1_input, bc2_input, t_input])
            res_vy[p,i+1-snaps_crop[0]:i+window_size+1-snaps_crop[0]] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
        else:
            diff = Nt_snaps - i - 1
            par_input = b_par_data_vy[p, i+1:i+1+diff, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
            bc1_input = b_bc1_data_vy[p, i+1:i+1+diff, :] 
            bc2_input = b_bc2_data_vy[p, i+1:i+1+diff, :] 
            t_input = np.expand_dims(t_data_vy[0:diff],1)  

            prediction = mito_model_vy([par_input, ic_input, bc1_input, bc2_input, t_input])
            res_vy[p,i+1-snaps_crop[0]:i+diff+1-snaps_crop[0]] = prediction
            

res_vy = ls_scaler_vy.inverse_transform(res_vy.reshape(Np*Nt_snaps_crop,latent_dim_vy))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_vy = ae_model_vy.decoder(res_vy).numpy()
full_res_vy = np.squeeze(scaler_vy.inverse_transform(full_res_vy))

# Reshape data for analysis and visualization
full_res_r_vy = np.reshape(full_res_vy, (Np, Nt_snaps_crop, Nn))


#### Plot6

In [ ]:
x_lower = time[snaps_crop[0]+snaps[0]]/3600/24
x_upper = time[snaps_crop[1]+snaps[0]]/3600/24
time_crop = time[snaps_crop[0]+snaps[0]:snaps_crop[1]+snaps[0]]/3600/24

fig2, axs2 = plt.subplots(3, 1, figsize=(15,11), sharex=True, layout='constrained')

axs2[0].plot(time_crop,full_res_r_dep[1,:,sensors[0]],label='MITONet (sensor 1)', linewidth=2, linestyle='-',color='blue')
axs2[0].plot(time_crop,dataset_r_dep[1,snaps_crop[0]:snaps_crop[1],sensors[0]],label='AdH (sensor 1)', linewidth=2, linestyle='--',color='blue')
axs2[0].plot(time_crop,full_res_r_dep[1,:,sensors[1]],label='MITONet (sensor 2)', linewidth=2, linestyle='-',color='orange')
axs2[0].plot(time_crop,dataset_r_dep[1,snaps_crop[0]:snaps_crop[1],sensors[1]],label='AdH (sensor 2)', linewidth=2, linestyle='--',color='orange')
axs2[0].plot(time_crop,full_res_r_dep[1,:,sensors[2]],label='MITONet (sensor 3)', linewidth=2, linestyle='-',color='green')
axs2[0].plot(time_crop,dataset_r_dep[1,snaps_crop[0]:snaps_crop[1],sensors[2]],label='AdH (sensor 3)', linewidth=2, linestyle='--',color='green')
axs2[0].set_xlim(x_lower,x_upper)
axs2[0].set_ylabel('Water Surface Elevation  (m)')
axs2[0].legend(bbox_to_anchor=(0.5,1.2), loc='upper center', ncol = 6, columnspacing=0.5, fontsize=12)

axs2[1].plot(time_crop,full_res_r_vx[1,:,sensors[0]],label='MITONet (sensor 1)', linewidth=2, linestyle='-',color='blue')
axs2[1].plot(time_crop,dataset_r_vx[1,snaps_crop[0]:snaps_crop[1],sensors[0]],label='AdH (sensor 1)', linewidth=2, linestyle='--',color='blue')
axs2[1].plot(time_crop,full_res_r_vx[1,:,sensors[1]],label='MITONet (sensor 2)', linewidth=2, linestyle='-',color='orange')
axs2[1].plot(time_crop,dataset_r_vx[1,snaps_crop[0]:snaps_crop[1],sensors[1]],label='AdH (sensor 2)', linewidth=2, linestyle='--',color='orange')
axs2[1].plot(time_crop,full_res_r_vx[1,:,sensors[2]],label='MITONet (sensor 3)', linewidth=2, linestyle='-',color='green')
axs2[1].plot(time_crop,dataset_r_vx[1,snaps_crop[0]:snaps_crop[1],sensors[2]],label='AdH (sensor 3)', linewidth=2, linestyle='--',color='green')
axs2[1].set_xlim(x_lower,x_upper)
axs2[1].set_ylabel('U-Velocity  (m/s)')

axs2[2].plot(time_crop,full_res_r_vy[1,:,sensors[0]],label='MITONet (sensor 1)', linewidth=2, linestyle='-',color='blue')
axs2[2].plot(time_crop,dataset_r_vy[1,snaps_crop[0]:snaps_crop[1],sensors[0]],label='AdH (sensor 1)', linewidth=2, linestyle='--',color='blue')
axs2[2].plot(time_crop,full_res_r_vy[1,:,sensors[1]],label='MITONet (sensor 2)', linewidth=2, linestyle='-',color='orange')
axs2[2].plot(time_crop,dataset_r_vy[1,snaps_crop[0]:snaps_crop[1],sensors[1]],label='AdH (sensor 2)', linewidth=2, linestyle='--',color='orange')
axs2[2].plot(time_crop,full_res_r_vy[1,:,sensors[2]],label='MITONet (sensor 3)', linewidth=2, linestyle='-',color='green')
axs2[2].plot(time_crop,dataset_r_vy[1,snaps_crop[0]:snaps_crop[1],sensors[2]],label='AdH (sensor 3)', linewidth=2, linestyle='--',color='green')
axs2[2].set_xlim(x_lower,x_upper)
axs2[2].set_ylabel('V-Velocity  (m/s)')
axs2[2].set_xlabel('Time (days)')
axs2[2].set_xticks(np.arange(29.5,45,1))


In [ ]:
tstep1 = 0
tstep2 = tstep1 + int((0.25*24)*60/30)
tstep3 = tstep1 + int((1*24)*60/30)
tstep4 = tstep1 + int((5*24)*60/30)
param = 1

fig, axs = plt.subplots(3, 4, figsize=(20, 10), layout='constrained')#, sharey='col')

axs[0,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                   full_res_r_dep[param,tstep1,:], vmin=0, 
                   vmax=30, cmap='cmo.balance', shading='gouraud')
axs[0,0].axis('off')

axs[0,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                   full_res_r_dep[param,tstep2,:], vmin=0, 
                   vmax=30, cmap='cmo.balance', shading='gouraud')
axs[0,1].axis('off')

axs[0,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                   full_res_r_dep[param,tstep3,:], vmin=0, 
                   vmax=30, cmap='cmo.balance', shading='gouraud')
axs[0,2].axis('off')

last_dep = axs[0,3].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                           full_res_r_dep[param,tstep4,:], vmin=0, 
                           vmax=30, cmap='cmo.balance', shading='gouraud')
axs[0,3].axis('off')


axs[1,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                   full_res_r_vx[param,tstep1,:], vmin=-2, 
                   vmax=2, cmap='cmo.balance', shading='gouraud')
axs[1,0].axis('off')

axs[1,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                   full_res_r_vx[param,tstep2,:], vmin=-2, 
                   vmax=2, cmap='cmo.balance', shading='gouraud')
axs[1,1].axis('off')

axs[1,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                   full_res_r_vx[param,tstep3,:], vmin=-2, 
                   vmax=2, cmap='cmo.balance', shading='gouraud')
axs[1,2].axis('off')

last_vx = axs[1,3].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                           full_res_r_vx[param,tstep4,:], vmin=-2, 
                           vmax=2, cmap='cmo.balance', shading='gouraud')
axs[1,3].axis('off')

axs[2,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                   full_res_r_vy[param,tstep1,:], vmin=-2, 
                   vmax=2, cmap='cmo.balance', shading='gouraud')
axs[2,0].axis('off') 

axs[2,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                   full_res_r_vy[param,tstep2,:], vmin=-2, 
                   vmax=2, cmap='cmo.balance', shading='gouraud')
axs[2,1].axis('off')

axs[2,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                   full_res_r_vy[param,tstep3,:], vmin=-2, 
                   vmax=2, cmap='cmo.balance', shading='gouraud')
axs[2,2].axis('off') 

last_vy = axs[2,3].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles'], 
                           full_res_r_vy[param,tstep4,:], vmin=-2, 
                           vmax=2, cmap='cmo.balance', shading='gouraud')
axs[2,3].axis('off')

axs[0, 0].text(0.5, 1, 'Day {0}'.format(time[snaps_crop[0]+tstep1]), size=20, ha="center", va="center", 
               rotation=0, transform=axs[0,0].transAxes)
axs[0, 1].text(0.5, 1, 'Day {0}'.format(time[snaps_crop[0]+tstep2]), size=20, ha="center", va="center", 
               rotation=0, transform=axs[0,1].transAxes)
axs[0, 2].text(0.5, 1, 'Day {0}'.format(time[snaps_crop[0]+tstep3]), size=20, ha="center", va="center", 
               rotation=0, transform=axs[0,2].transAxes)
axs[0, 3].text(0.5, 1, 'Day {0}'.format(time[snaps_crop[0]+tstep4]), size=20, ha="center", va="center", 
               rotation=0, transform=axs[0,3].transAxes)

axs[0, 0].text(-0.1, 0.5, 'Water Surface Elevation', size=20, ha="center", va="center", 
               rotation=90, transform=axs[0,0].transAxes)
axs[1, 0].text(-0.1, 0.5, 'U-Velocity', size=20, ha="center", va="center", 
               rotation=90, transform=axs[1,0].transAxes)
axs[2, 0].text(-0.1, 0.5, 'V-Velocity', size=20, ha="center", va="center", 
               rotation=90, transform=axs[2,0].transAxes)

cbar = plt.colorbar(last_dep, orientation='vertical', aspect=15)
cbar.set_label(label='(cm)', size=20) 
cbar = plt.colorbar(last_vx, orientation='vertical', aspect=15)
cbar.set_label(label='(cm/s)', size=20) 
cbar = plt.colorbar(last_vy, orientation='vertical', aspect=15)
cbar.set_label(label='(cm/s)',  size=20) 